# 01 — Cohort Retention Analysis

## Business Problem

A digital wallet company wants to understand whether its customer base is healthy before investing more money into acquisition and retention campaigns.

The business questions are:

1. Which customer groups are still active?
2. Which segments show signs of churn or dormancy?
3. Which users are valuable but at risk?
4. Can we build a retention baseline before moving into LTV, A/B testing, uplift modeling, and causal inference?

## Important Dataset Limitation

The available dataset is a customer-level snapshot. It does **not** include:

- signup date
- transaction-level timestamps
- campaign exposure
- treatment/control assignment

Therefore, this notebook does **not** perform true calendar cohort analysis. Instead, it builds **retention proxy cohorts** using activity, recency, transaction intensity, app usage, income level, and payment method.

This limitation is intentionally documented because forcing a method onto unsuitable data is a common analytics pitfall.

## 1. Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
REPORTS_DIR = Path("../reports")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = RAW_DIR / "digital_wallet_ltv_dataset.csv"

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}. "
        "Place digital_wallet_ltv_dataset.csv under data/raw/."
    )

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
df.head()

## 2. Data Quality Check

Before analyzing retention, we need to understand whether the dataset is reliable enough for customer analytics.

Checks covered here:

- schema
- missing values
- duplicate customers
- basic numeric distributions
- categorical cardinality

In [ ]:
display(df.info())
display(df.describe(include="all").T)

In [ ]:
missing = (
    df.isna()
    .mean()
    .sort_values(ascending=False)
    .to_frame("missing_rate")
)

display(missing)

if "Customer_ID" in df.columns:
    duplicate_customers = df.duplicated("Customer_ID").sum()
    print("Duplicate Customer_ID rows:", duplicate_customers)
else:
    print("Customer_ID column not found.")

In [ ]:
categorical_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()

print("Categorical columns:", categorical_cols)
print("Numeric columns:", numeric_cols)

for col in categorical_cols:
    print(f"\n{col}")
    display(df[col].value_counts(dropna=False).head(20))

## 3. Retention Proxy Design

Because we do not have true activity dates, we define practical retention proxies.

Definitions:

- `retained_30d`: customer had a transaction in the last 30 days
- `dormant_90d`: customer had no transaction for more than 90 days
- `recency_bucket`: customer grouped by `Last_Transaction_Days_Ago`
- `activity_cohort`: customer grouped by `Active_Days`
- `transaction_cohort`: customer grouped by `Total_Transactions`

These proxies are not perfect, but they are useful for baseline customer analytics.

In [ ]:
df = df.copy()

df["retained_30d"] = (df["Last_Transaction_Days_Ago"] <= 30).astype(int)
df["dormant_90d"] = (df["Last_Transaction_Days_Ago"] > 90).astype(int)

df["recency_bucket"] = pd.cut(
    df["Last_Transaction_Days_Ago"],
    bins=[-1, 7, 30, 90, np.inf],
    labels=["0-7 days", "8-30 days", "31-90 days", "90+ days"],
)

df["activity_cohort"] = pd.qcut(
    df["Active_Days"].rank(method="first"),
    q=4,
    labels=[
        "Q1_low_activity",
        "Q2_mid_low_activity",
        "Q3_mid_high_activity",
        "Q4_high_activity",
    ],
)

df["transaction_cohort"] = pd.qcut(
    df["Total_Transactions"].rank(method="first"),
    q=4,
    labels=[
        "Q1_low_transactions",
        "Q2_mid_low_transactions",
        "Q3_mid_high_transactions",
        "Q4_high_transactions",
    ],
)

df[
    [
        "Customer_ID",
        "Active_Days",
        "Last_Transaction_Days_Ago",
        "retained_30d",
        "dormant_90d",
        "recency_bucket",
        "activity_cohort",
        "transaction_cohort",
    ]
].head()

## 4. Overall Retention Health

This section gives the first executive-level health check.

In [ ]:
overall_summary = pd.DataFrame(
    {
        "metric": [
            "customers",
            "retained_30d_rate",
            "dormant_90d_rate",
            "avg_ltv",
            "median_ltv",
            "avg_total_spent",
            "avg_active_days",
            "avg_last_transaction_days_ago",
        ],
        "value": [
            len(df),
            df["retained_30d"].mean(),
            df["dormant_90d"].mean(),
            df["LTV"].mean(),
            df["LTV"].median(),
            df["Total_Spent"].mean(),
            df["Active_Days"].mean(),
            df["Last_Transaction_Days_Ago"].mean(),
        ],
    }
)

display(overall_summary)

In [ ]:
plt.figure(figsize=(8, 4))
df["Last_Transaction_Days_Ago"].hist(bins=40)
plt.title("Distribution of Recency: Last Transaction Days Ago")
plt.xlabel("Days since last transaction")
plt.ylabel("Customer count")
plt.show()

plt.figure(figsize=(8, 4))
df["Active_Days"].hist(bins=40)
plt.title("Distribution of Active Days")
plt.xlabel("Active days")
plt.ylabel("Customer count")
plt.show()

## 5. Retention by Activity Cohort

Hypothesis:

> Customers with more active days should have stronger recent retention and higher LTV.

This is not causal. It is a descriptive relationship.

In [ ]:
activity_summary = (
    df.groupby("activity_cohort", observed=False)
    .agg(
        customers=("Customer_ID", "count"),
        retained_30d_rate=("retained_30d", "mean"),
        dormant_90d_rate=("dormant_90d", "mean"),
        avg_ltv=("LTV", "mean"),
        median_ltv=("LTV", "median"),
        avg_total_spent=("Total_Spent", "mean"),
        avg_satisfaction=("Customer_Satisfaction_Score", "mean"),
        avg_transactions=("Total_Transactions", "mean"),
    )
    .reset_index()
)

display(activity_summary)

In [ ]:
plt.figure(figsize=(9, 4))
plt.bar(
    activity_summary["activity_cohort"].astype(str),
    activity_summary["retained_30d_rate"],
)
plt.title("30-Day Retention Proxy by Activity Cohort")
plt.ylabel("Retention rate")
plt.xticks(rotation=30)
plt.show()

plt.figure(figsize=(9, 4))
plt.bar(
    activity_summary["activity_cohort"].astype(str),
    activity_summary["avg_ltv"],
)
plt.title("Average LTV by Activity Cohort")
plt.ylabel("Average LTV")
plt.xticks(rotation=30)
plt.show()

## 6. Retention by Business Segments

We inspect retention and LTV by:

- income level
- preferred payment method
- app usage frequency
- location

This helps identify commercially relevant customer groups.

In [ ]:
def segment_summary(data, segment_col):
    summary = (
        data.groupby(segment_col, observed=False)
        .agg(
            customers=("Customer_ID", "count"),
            retained_30d_rate=("retained_30d", "mean"),
            dormant_90d_rate=("dormant_90d", "mean"),
            avg_ltv=("LTV", "mean"),
            median_ltv=("LTV", "median"),
            avg_transactions=("Total_Transactions", "mean"),
            avg_total_spent=("Total_Spent", "mean"),
            avg_satisfaction=("Customer_Satisfaction_Score", "mean"),
            avg_support_tickets=("Support_Tickets_Raised", "mean"),
        )
        .sort_values("avg_ltv", ascending=False)
    )
    return summary

for col in [
    "Income_Level",
    "Preferred_Payment_Method",
    "App_Usage_Frequency",
    "Location",
]:
    if col in df.columns:
        print(f"\n===== Segment: {col} =====")
        display(segment_summary(df, col))

## 7. RFM-Style Customer Profiling

RFM means:

- Recency: how recently the customer transacted
- Frequency: how often the customer transacted
- Monetary: how much the customer spent

This is a strong classical customer analytics baseline.

In [ ]:
df["recency_score"] = pd.qcut(
    (-df["Last_Transaction_Days_Ago"]).rank(method="first"),
    q=5,
    labels=[1, 2, 3, 4, 5],
).astype(int)

df["frequency_score"] = pd.qcut(
    df["Total_Transactions"].rank(method="first"),
    q=5,
    labels=[1, 2, 3, 4, 5],
).astype(int)

df["monetary_score"] = pd.qcut(
    df["Total_Spent"].rank(method="first"),
    q=5,
    labels=[1, 2, 3, 4, 5],
).astype(int)

df["rfm_score"] = (
    df["recency_score"] + df["frequency_score"] + df["monetary_score"]
)

df["rfm_segment"] = pd.cut(
    df["rfm_score"],
    bins=[2, 6, 10, 13, 15],
    labels=[
        "At risk / low value",
        "Developing",
        "Strong",
        "Champions",
    ],
)

rfm_summary = (
    df.groupby("rfm_segment", observed=False)
    .agg(
        customers=("Customer_ID", "count"),
        avg_ltv=("LTV", "mean"),
        median_ltv=("LTV", "median"),
        retained_30d_rate=("retained_30d", "mean"),
        dormant_90d_rate=("dormant_90d", "mean"),
        avg_rfm_score=("rfm_score", "mean"),
    )
)

display(rfm_summary)

In [ ]:
plt.figure(figsize=(8, 4))
rfm_summary["avg_ltv"].plot(kind="bar")
plt.title("Average LTV by RFM Segment")
plt.ylabel("Average LTV")
plt.xticks(rotation=30)
plt.show()

plt.figure(figsize=(8, 4))
rfm_summary["dormant_90d_rate"].plot(kind="bar")
plt.title("Dormant 90-Day Rate by RFM Segment")
plt.ylabel("Dormant rate")
plt.xticks(rotation=30)
plt.show()

## 8. High-Value but At-Risk Customers

This is the most business-relevant retention segment.

Definition:

- high value: top 25% by LTV
- at risk: last transaction more than 90 days ago

These customers may be good candidates for retention campaigns. However, this notebook cannot prove whether an incentive would cause them to return. That requires A/B testing or uplift modeling.

In [ ]:
ltv_75 = df["LTV"].quantile(0.75)

df["high_value"] = (df["LTV"] >= ltv_75).astype(int)
df["high_value_at_risk"] = (
    (df["high_value"] == 1) & (df["dormant_90d"] == 1)
).astype(int)

hv_risk_summary = pd.DataFrame(
    {
        "metric": [
            "high_value_customers",
            "high_value_at_risk_customers",
            "share_high_value_at_risk",
            "avg_ltv_high_value_at_risk",
            "total_ltv_high_value_at_risk",
        ],
        "value": [
            df["high_value"].sum(),
            df["high_value_at_risk"].sum(),
            df["high_value_at_risk"].mean(),
            df.loc[df["high_value_at_risk"] == 1, "LTV"].mean(),
            df.loc[df["high_value_at_risk"] == 1, "LTV"].sum(),
        ],
    }
)

display(hv_risk_summary)

display(
    df.loc[df["high_value_at_risk"] == 1]
    .sort_values("LTV", ascending=False)
    .head(20)
)

## 9. Risk Matrix: LTV x Recency

This matrix helps prioritize retention strategy.

Interpretation:

- High LTV + recent activity: protect and nurture
- High LTV + dormant: retention opportunity
- Low LTV + dormant: usually lower priority
- Low LTV + recent activity: develop or upsell

In [ ]:
df["ltv_quartile"] = pd.qcut(
    df["LTV"].rank(method="first"),
    q=4,
    labels=["Q1_low_ltv", "Q2_mid_low_ltv", "Q3_mid_high_ltv", "Q4_high_ltv"],
)

risk_matrix_count = pd.crosstab(
    df["ltv_quartile"],
    df["recency_bucket"],
    values=df["Customer_ID"],
    aggfunc="count",
)

risk_matrix_share = pd.crosstab(
    df["ltv_quartile"],
    df["recency_bucket"],
    values=df["Customer_ID"],
    aggfunc="count",
    normalize="index",
)

display(risk_matrix_count)
display(risk_matrix_share)

In [ ]:
plt.figure(figsize=(8, 4))
plt.imshow(risk_matrix_share, aspect="auto")
plt.xticks(range(len(risk_matrix_share.columns)), risk_matrix_share.columns, rotation=30)
plt.yticks(range(len(risk_matrix_share.index)), risk_matrix_share.index)
plt.title("Recency Distribution Within LTV Quartiles")
plt.colorbar(label="Share within LTV quartile")
plt.show()

## 10. Pitfalls and Interpretation

### Pitfall 1: Calling this true cohort retention

This is not true cohort retention because we do not have signup month or transaction month.

### Pitfall 2: Treating descriptive patterns as causal

If high activity customers have higher LTV, it does not necessarily mean increasing activity will cause higher LTV.

### Pitfall 3: Using future-looking variables carelessly

Variables such as `Total_Spent`, `Loyalty_Points_Earned`, and `Cashback_Received` may be accumulated over the customer lifetime. They can create leakage in predictive modeling.

### Pitfall 4: Targeting all at-risk customers

A good retention strategy should consider both customer value and likelihood of incremental response.

## 11. Save Processed Dataset and Report

In [ ]:
processed_path = PROCESSED_DIR / "wallet_retention_baseline.csv"
df.to_csv(processed_path, index=False)

report = f'''
# 01 Cohort Retention Analysis Summary

## Dataset limitation
The dataset is customer-level and does not include signup dates or transaction-level timestamps.
Therefore, this notebook uses retention proxies rather than true calendar cohorts.

## Retention proxy definitions
- retained_30d = Last_Transaction_Days_Ago <= 30
- dormant_90d = Last_Transaction_Days_Ago > 90
- activity_cohort = quartiles of Active_Days
- transaction_cohort = quartiles of Total_Transactions
- rfm_segment = Recency + Frequency + Monetary score

## Key baseline metrics
- Total customers: {len(df)}
- 30-day retained rate: {df["retained_30d"].mean():.4f}
- 90-day dormant rate: {df["dormant_90d"].mean():.4f}
- Average LTV: {df["LTV"].mean():.2f}
- High-value at-risk customers: {df["high_value_at_risk"].sum()}

## Business interpretation
This notebook identifies retention risk and customer value segments.
It does not estimate causal campaign impact. That will be handled later using A/B testing and uplift modeling.
'''

report_path = REPORTS_DIR / "01_cohort_retention_summary.md"
report_path.write_text(report)

print("Saved processed data to:", processed_path)
print("Saved report to:", report_path)

## 12. Interview Discussion Questions

Use these to practice explaining the notebook:

1. Why is this not true cohort analysis?
2. How did you define retention without transaction timestamps?
3. What is the risk of using `Total_Spent` in future LTV prediction?
4. Which segment would you prioritize for a retention campaign?
5. Why do we need A/B testing or uplift modeling before giving incentives?